In [1]:
# Librerias

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import r2_score

from transformers import pipeline
from src.nlp import extract_emotions

import joblib
import os

c:\Users\xaren\OneDrive\Documentos\Angel Sotelo\Projects\app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 36387.83it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
df = pd.read_csv("songs_dataset_completo_final.csv")

print(df.shape)
df.head()


(688, 15)


,artist,song,lyrics,surprise,joy,neutral,sadness,fear,anger,disgust,key,mode,tempo,energy,valence
0,Taylor Swift,Blank Space,Nice to meet you\nWhere you been?\nI could sho...,0.881859,0.033813,0.030993,0.021404,0.014393,0.010976,0.006561,5,1,96.01,0.68,0.58
1,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah)\n\nFever dream high i...",0.013157,0.016586,0.145919,0.029140,0.260845,0.288338,0.246014,9,1,169.99,0.70,0.56
2,Taylor Swift,Shake It Off,I stay out too late\nGot nothing in my brain\n...,0.014898,0.006023,0.190369,0.036084,0.639317,0.084702,0.028606,7,1,160.02,0.79,0.94
3,Taylor Swift,Style,"Midnight\nYou come and pick me up, no headligh...",0.023875,0.006980,0.099869,0.034349,0.540428,0.275398,0.019101,2,1,95.02,0.79,0.46
4,Taylor Swift,cardigan,"Vintage tee, brand-new phone\nHigh heels on co...",0.120870,0.022434,0.364791,0.021662,0.178900,0.022693,0.268651,0,0,130.03,0.58,0.55


In [3]:
# eliminar filas sin audio features
df = df.dropna(subset=["valence", "energy", "mode","key"])

# asegurar tipos
df["mode"] = df["mode"].astype(int)

print("Filas finales:", len(df))

Filas finales: 688


In [4]:
emotion_cols = [
    "surprise", "neutral", "disgust",
    "anger", "joy", "fear", "sadness"
]

X = df[emotion_cols]

y_valence = df["valence"]
y_energy = df["energy"]
y_mode = df["mode"]
y_key = df["key"]  # valores 0–11

In [5]:
X_train, X_test, y_val_train, y_val_test = train_test_split(
    X, y_valence, test_size=0.2, random_state=42
)

_, _, y_energy_train, y_energy_test = train_test_split(
    X, y_energy, test_size=0.2, random_state=42
)

_, _, y_mode_train, y_mode_test = train_test_split(
    X, y_mode, test_size=0.2, random_state=42
)

_, _, y_key_train, y_key_test = train_test_split(
    X, y_key, test_size=0.2, random_state=42
)

In [6]:

model_valence = RandomForestRegressor(n_estimators=100, random_state=42)
model_valence.fit(X_train, y_val_train)

val_pred = model_valence.predict(X_test)

print("🎯 VALENCE")
print("RMSE:", np.sqrt(mean_squared_error(y_val_test, val_pred)))
print("R2:", r2_score(y_val_test, val_pred))

🎯 VALENCE
RMSE: 0.21431804413365899
R2: 0.04197623503416781


In [7]:
model_energy = RandomForestRegressor(n_estimators=100, random_state=42)
model_energy.fit(X_train, y_energy_train)

energy_pred = model_energy.predict(X_test)

print("\n⚡ ENERGY")
print("RMSE:", np.sqrt(mean_squared_error(y_energy_test, energy_pred)))
print("R2:", r2_score(y_energy_test, energy_pred))


⚡ ENERGY
RMSE: 0.21670248280957738
R2: -0.0038550955147333976


In [8]:

model_mode = RandomForestClassifier(n_estimators=100, random_state=42)
model_mode.fit(X_train, y_mode_train)

mode_pred = model_mode.predict(X_test)

print("\n🎵 MODE")
print("Accuracy:", accuracy_score(y_mode_test, mode_pred))
print(classification_report(y_mode_test, mode_pred))


🎵 MODE
Accuracy: 0.6014492753623188
              precision    recall  f1-score   support

           0       0.53      0.47      0.50        58
           1       0.64      0.70      0.67        80

    accuracy                           0.60       138
   macro avg       0.59      0.58      0.58       138
weighted avg       0.60      0.60      0.60       138



In [9]:
from sklearn.ensemble import RandomForestClassifier

model_key = RandomForestClassifier(n_estimators=200, random_state=42)
model_key.fit(X_train, y_key_train)

key_pred = model_key.predict(X_test)

print("🎼 KEY MODEL")
print("Accuracy:", accuracy_score(y_key_test, key_pred))
print(classification_report(y_key_test, key_pred))

🎼 KEY MODEL
Accuracy: 0.08695652173913043
              precision    recall  f1-score   support

           0       0.04      0.08      0.05        12
           1       0.00      0.00      0.00        17
           2       0.00      0.00      0.00        11
           3       0.00      0.00      0.00         3
           4       0.18      0.17      0.17        12
           5       0.00      0.00      0.00        12
           6       0.23      0.43      0.30         7
           7       0.07      0.06      0.06        18
           8       0.00      0.00      0.00         6
           9       0.22      0.22      0.22        18
          10       0.00      0.00      0.00         5
          11       0.09      0.06      0.07        17

    accuracy                           0.09       138
   macro avg       0.07      0.08      0.07       138
weighted avg       0.08      0.09      0.08       138



In [10]:
feature_columns = X.columns.tolist()

In [ ]:
def predict_song_profile(emotion_scores):

    X_input = pd.DataFrame([emotion_scores])

    #
    X_input = X_input.reindex(columns=feature_columns, fill_value=0)

    valence = model_valence.predict(X_input)[0]
    energy = model_energy.predict(X_input)[0]
    mode = model_mode.predict(X_input)[0]

    return {
        "valence": round(valence, 2),
        "energy": round(energy, 2),
        "mode": "major" if mode == 1 else "minor"
    }

In [12]:
def interpret_profile(valence, energy):
    if valence > 0.6 and energy > 0.6:
        return "Happy & Energetic 🔥"
    elif valence < 0.4 and energy < 0.4:
        return "Sad & Calm 😢"
    elif energy > 0.6:
        return "Intense ⚡"
    else:
        return "Chill 🎧"

In [13]:
key_map = {
    0: "C", 1: "C#", 2: "D", 3: "D#", 4: "E", 5: "F",
    6: "F#", 7: "G", 8: "G#", 9: "A", 10: "A#", 11: "B"
}

In [14]:
def predict_key(emotion_scores):

    X_input = pd.DataFrame([emotion_scores])

    # 🔥 CLAVE
    X_input = X_input.reindex(columns=feature_columns, fill_value=0)

    probs = model_key.predict_proba(X_input)[0]
    top_idx = np.argsort(probs)[-3:][::-1]

    result = []
    for i in top_idx:
        result.append({
            "key": key_map[i],
            "confidence": round(probs[i], 2)
        })

    return result

In [15]:
def analyze_lyrics(text):
    # 1. emociones
    emotions = extract_emotions(text)

    # 2. predicción
    profile = predict_song_profile(emotions)

    # 3. interpretación
    mood = interpret_profile(profile["valence"], profile["energy"])

    # 4. Key
    keys = predict_key(emotions)

    return {
        "emotions": emotions,
        "valence": profile["valence"],
        "energy": profile["energy"],
        "mode": profile["mode"],
        "mood": mood,
        "key_suggestions": keys
    }

In [ ]:
import joblib
import os

#  Crear carpeta models si no existe
os.makedirs("models", exist_ok=True)

#  Guardar modelos
joblib.dump(model_valence, "models/model_valence.pkl")
joblib.dump(model_energy, "models/model_energy.pkl")
joblib.dump(model_mode, "models/model_mode.pkl")
joblib.dump(model_key, "models/model_key.pkl")

#  Guardar columnas de features (CRÍTICO 🔥)
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, "models/features.pkl")

print("Modelos y features guardados correctamente en /models")

Modelos y features guardados correctamente en /models
